# Week 13 — Poisson Regression and Count Data Models
## Modelos de Análisis Estadístico | Universidad de los Andes
### Prof. Alejandra Tabares

---

## 1. When to Use Poisson Regression

Poisson regression is the natural starting point whenever the **outcome variable is a non-negative integer count**:

- Number of doctor visits in a year
- Number of insurance claims per policyholder
- Number of traffic accidents at an intersection per month
- Number of publications by a researcher
- Number of defects in a manufactured unit

Ordinary linear regression is inappropriate for counts because:
1. It can predict **negative values**, which are impossible for counts.
2. The **error variance** for counts increases with the mean (heteroscedasticity), violating OLS assumptions.
3. Count distributions are typically **right-skewed**, especially for rare events.

### Assumptions of Poisson Regression

| Assumption | Description |
|---|---|
| **Poisson distribution** | $Y_i \sim \text{Poisson}(\mu_i)$ — counts are non-negative integers |
| **Log-linearity** | $\log(\mu_i)$ is a linear function of the predictors |
| **Independence** | Observations are independent of each other |
| **Equidispersion** | $E[Y_i] = \text{Var}(Y_i) = \mu_i$ — the most frequently violated assumption |

---

## 2. The Poisson Regression Model

Poisson regression is a **Generalized Linear Model (GLM)** with three components:

**Random component:**
$$Y_i \sim \text{Poisson}(\mu_i), \qquad i = 1, \ldots, n$$

**Systematic component (linear predictor):**
$$\eta_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \ldots + \beta_p x_{ip}$$

**Link function (log link):**
$$\log(\mu_i) = \beta_0 + \beta_1 x_{i1} + \ldots + \beta_p x_{ip}$$

which equivalently gives the **mean response** as:
$$\mu_i = \exp(\beta_0 + \beta_1 x_{i1} + \ldots + \beta_p x_{ip})$$

The parameters $\boldsymbol{\beta}$ are estimated by **Maximum Likelihood Estimation (MLE)**. The log-likelihood is:
$$\ell(\boldsymbol{\beta}) = \sum_{i=1}^n \left[ y_i \log(\mu_i) - \mu_i - \log(y_i!) \right]$$

No closed-form solution exists; the score equations are solved iteratively using **Iteratively Reweighted Least Squares (IRLS)**.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

# Reproducibility
np.random.seed(42)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('Libraries loaded successfully.')

---

## 3. Simulating Count Data

We simulate a dataset of **500 individuals** where the outcome is the **number of doctor visits per year**. The data-generating process mirrors a Poisson regression:

$$\log(\mu_i) = 0.8 + 0.025 \cdot \text{age}_i - 0.4 \cdot \text{income\_high}_i$$

- `age`: continuous, centered around 45 years.
- `income_high`: binary indicator (1 = high income, 0 = low income).
- `visits`: Poisson-distributed count outcome.

In [ ]:
n = 500

# Predictors
age = np.random.normal(loc=45, scale=12, size=n).clip(18, 80)
income_high = np.random.binomial(1, 0.4, size=n)  # 40% high income

# True linear predictor
log_mu = 0.8 + 0.025 * age - 0.4 * income_high
mu = np.exp(log_mu)

# Poisson-distributed counts
visits = np.random.poisson(lam=mu)

# Assemble DataFrame
df = pd.DataFrame({'visits': visits, 'age': age, 'income_high': income_high})

print(df.head(10))
print('\nDescriptive statistics:')
print(df.describe().round(3))

In [ ]:
# Quick look at the count distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of counts
axes[0].hist(df['visits'], bins=range(0, df['visits'].max() + 2),
             edgecolor='white', color='steelblue', align='left')
axes[0].set(title='Distribution of Doctor Visits', xlabel='Number of visits',
            ylabel='Frequency')

# Mean vs Variance check
mean_v = df['visits'].mean()
var_v = df['visits'].var()
axes[1].bar(['Mean', 'Variance'], [mean_v, var_v], color=['steelblue', 'coral'])
axes[1].set(title='Mean vs Variance of Visits',
            ylabel='Value')
for i, val in enumerate([mean_v, var_v]):
    axes[1].text(i, val + 0.1, f'{val:.2f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()
print(f'\nMean = {mean_v:.3f},  Variance = {var_v:.3f}')
print(f'Variance / Mean = {var_v / mean_v:.3f}  (ideal ≈ 1 for Poisson)')

---

## 4. Fitting Poisson Regression with `statsmodels`

We use `statsmodels.api.GLM` with `family=sm.families.Poisson()`. The default link for the Poisson family is the **log link**.

In [ ]:
# Fit Poisson GLM
poisson_model = smf.glm(
    formula='visits ~ age + income_high',
    data=df,
    family=sm.families.Poisson()
).fit()

print(poisson_model.summary())

---

## 5. Interpreting Coefficients as Rate Ratios

The coefficients from Poisson regression are on the **log scale**. To interpret them in the original count scale, we exponentiate:

$$\text{Rate Ratio (IRR)} = e^{\hat{\beta}_j}$$

**Interpretation rules:**

- $e^{\hat{\beta}_j} > 1$: a one-unit increase in $x_j$ **multiplies** the expected count by $e^{\hat{\beta}_j}$ (rate increases).
- $e^{\hat{\beta}_j} < 1$: a one-unit increase in $x_j$ **reduces** the expected count (rate decreases).
- $e^{\hat{\beta}_j} = 1$: no effect.

The **percent change** in the expected count per unit increase in $x_j$ is $(e^{\hat{\beta}_j} - 1) \times 100\%$.

In [ ]:
# Extract coefficients and compute Rate Ratios with 95% CI
coef = poisson_model.params
ci = poisson_model.conf_int()

irr_table = pd.DataFrame({
    'Coefficient (log scale)': coef,
    'Rate Ratio (IRR)': np.exp(coef),
    'IRR CI lower (95%)': np.exp(ci[0]),
    'IRR CI upper (95%)': np.exp(ci[1]),
    'p-value': poisson_model.pvalues
})

print(irr_table.round(4))
print()
print('True values used in simulation:')
print('  Intercept     β₀ = 0.800  → exp(β₀) = {:.3f}'.format(np.exp(0.800)))
print('  age           β₁ = 0.025  → IRR = {:.3f}  (each extra year → +{:.1f}% visits)'.format(
      np.exp(0.025), (np.exp(0.025) - 1) * 100))
print('  income_high   β₂ = -0.400 → IRR = {:.3f}  (high income → {:.1f}% fewer visits)'.format(
      np.exp(-0.400), (np.exp(-0.400) - 1) * 100))

In [ ]:
# Visualise Rate Ratios with confidence intervals (forest plot style)
irr_plot = irr_table.drop('Intercept').copy()

fig, ax = plt.subplots(figsize=(7, 3))
y_pos = range(len(irr_plot))

ax.scatter(irr_plot['Rate Ratio (IRR)'], y_pos, color='steelblue', zorder=3, s=80)
for i, (idx, row) in enumerate(irr_plot.iterrows()):
    ax.hlines(i, row['IRR CI lower (95%)'], row['IRR CI upper (95%)'],
              color='steelblue', linewidth=2)

ax.axvline(1, color='red', linestyle='--', linewidth=1, label='IRR = 1 (no effect)')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(irr_plot.index)
ax.set(xlabel='Incidence Rate Ratio (IRR)', title='Rate Ratios with 95% CI — Poisson Regression')
ax.legend()
plt.tight_layout()
plt.show()

---

## 6. Offset Variables — Modeling Rates

Sometimes counts are recorded over **different exposure periods** (time, population size, area, etc.). For example:

- Accidents per hospital, but hospitals vary in the number of beds.
- Crime counts per city, but cities differ in population.
- Insurance claims per policyholder, but contracts span different durations.

To model the **rate per unit of exposure**, we include an **offset** — the log of the exposure — as a predictor with a fixed coefficient of 1:

$$\log(\mu_i) = \log(t_i) + \beta_0 + \beta_1 x_{i1} + \ldots$$

which is equivalent to modelling the **rate** $\lambda_i = \mu_i / t_i$:

$$\log\!\left(\frac{\mu_i}{t_i}\right) = \beta_0 + \beta_1 x_{i1} + \ldots$$

In `statsmodels` the offset is passed as `exposure` (which the library log-transforms automatically) or as `offset` (log-transformed manually).

In [ ]:
# ── Example with offset: insurance claims per policy-year ──────────────────
np.random.seed(7)
n_ins = 400

# Exposure: policy duration in years (varies from 0.5 to 5)
exposure = np.random.uniform(0.5, 5.0, size=n_ins)

# Predictor: driver age group (0 = young, 1 = senior)
senior = np.random.binomial(1, 0.35, size=n_ins)

# True rate model:  log(rate) = -1.5 + 0.6*senior
log_rate = -1.5 + 0.6 * senior
mu_ins = np.exp(log_rate) * exposure   # expected claims = rate * exposure

claims = np.random.poisson(mu_ins)

df_ins = pd.DataFrame({'claims': claims, 'exposure': exposure, 'senior': senior})

print('Insurance dataset (first 8 rows):')
print(df_ins.head(8).to_string(index=False))
print(f'\nAverage claims: {claims.mean():.2f},  Average exposure: {exposure.mean():.2f} yr')

In [ ]:
# Fit Poisson with exposure offset
poisson_offset = smf.glm(
    formula='claims ~ senior',
    data=df_ins,
    family=sm.families.Poisson(),
    exposure=df_ins['exposure']   # statsmodels takes log internally
).fit()

print(poisson_offset.summary())

print('\nRate Ratios:')
print(pd.DataFrame({
    'IRR': np.exp(poisson_offset.params),
    'CI lower': np.exp(poisson_offset.conf_int()[0]),
    'CI upper': np.exp(poisson_offset.conf_int()[1]),
}).round(4))
print('\nTrue IRR for senior: {:.3f}'.format(np.exp(0.6)))

---

## 7. Overdispersion: Diagnosis

**Overdispersion** occurs when the observed variance of the count outcome is greater than what the Poisson model predicts (i.e., greater than the mean). It is one of the most common violations in count data models.

### Causes
- Unobserved heterogeneity (omitted variables).
- Excess zeros (zero-inflation).
- Clustering or correlation among observations.
- Contagion effects (one event makes future events more likely).

### Consequences of Ignoring Overdispersion
- Standard errors are **underestimated**, leading to spuriously significant p-values.
- Hypothesis tests and confidence intervals are **invalid**.

### The Dispersion Statistic

A simple check uses the **Pearson chi-squared statistic** divided by the degrees of freedom:

$$\hat{\phi} = \frac{\chi^2_P}{n - p} = \frac{\sum_{i=1}^n (y_i - \hat{\mu}_i)^2 / \hat{\mu}_i}{n - p}$$

- $\hat{\phi} \approx 1$: equidispersion — Poisson is appropriate.
- $\hat{\phi} > 1$: overdispersion — consider Negative Binomial or quasi-Poisson.
- $\hat{\phi} < 1$: underdispersion (rare).

In [ ]:
# ── Simulate overdispersed data (Negative Binomial DGP) ───────────────────
np.random.seed(99)
n_od = 600

x1 = np.random.normal(0, 1, n_od)
x2 = np.random.binomial(1, 0.5, n_od)

# True log-mean
log_mu_od = 1.2 + 0.5 * x1 - 0.3 * x2
mu_od = np.exp(log_mu_od)

# Generate overdispersed counts (Negative Binomial with dispersion α=0.8)
alpha_true = 0.8
r = 1.0 / alpha_true          # shape parameter
p_nb = r / (r + mu_od)        # success probability
y_od = np.random.negative_binomial(r, p_nb)

df_od = pd.DataFrame({'y': y_od, 'x1': x1, 'x2': x2})

print(f'Mean:     {y_od.mean():.3f}')
print(f'Variance: {y_od.var():.3f}')
print(f'Variance / Mean (raw): {y_od.var() / y_od.mean():.3f}  (> 1 → overdispersed)')

In [ ]:
# Fit Poisson to the overdispersed data
poisson_od = smf.glm('y ~ x1 + x2', data=df_od,
                     family=sm.families.Poisson()).fit()

# Compute Pearson dispersion statistic
mu_hat = poisson_od.mu
pearson_chi2 = np.sum((y_od - mu_hat)**2 / mu_hat)
df_resid = poisson_od.df_resid
phi_hat = pearson_chi2 / df_resid

print(f'Pearson chi² / df  = {pearson_chi2:.2f} / {df_resid} = {phi_hat:.3f}')
print()
if phi_hat > 1.5:
    print('Dispersion statistic >> 1 → Strong evidence of overdispersion.')
    print('The Poisson model underestimates standard errors.')
elif phi_hat > 1.1:
    print('Mild overdispersion detected. Consider Negative Binomial or quasi-Poisson.')
else:
    print('No strong evidence of overdispersion. Poisson model may be adequate.')

# Also shown in the model summary as 'Pearson chi2'
print()
print('Note: statsmodels also reports the Pearson chi² in the summary table.')

---

## 8. Negative Binomial Regression

The **Negative Binomial (NB2)** model extends Poisson regression by adding a **dispersion parameter** $\alpha \geq 0$:

$$\text{Var}(Y_i) = \mu_i + \alpha \mu_i^2$$

When $\alpha = 0$ the model reduces to Poisson. As $\alpha$ increases, the distribution becomes more spread out, accommodating overdispersion.

The link function and linear predictor are **identical** to Poisson regression, so coefficient interpretation as rate ratios remains the same.

In `statsmodels` we use `sm.families.NegativeBinomial()` for the GLM interface, or `smf.negativebinomial()` which estimates $\alpha$ jointly with $\boldsymbol{\beta}$ via MLE.

In [ ]:
# Fit Negative Binomial regression
nb_model = smf.negativebinomial('y ~ x1 + x2', data=df_od).fit(disp=False)
print(nb_model.summary())

In [ ]:
# Compare Poisson vs Negative Binomial coefficients and standard errors
comparison = pd.DataFrame({
    'Poisson coef': poisson_od.params,
    'Poisson SE': poisson_od.bse,
    'NB coef': nb_model.params.drop('alpha', errors='ignore').reindex(poisson_od.params.index),
    'NB SE': nb_model.bse.drop('alpha', errors='ignore').reindex(poisson_od.params.index),
})

print('Coefficient comparison — Poisson vs Negative Binomial:')
print(comparison.round(4))
print()
print(f'Estimated alpha (dispersion): {nb_model.params["alpha"]:.4f}')
print(f'True alpha used in simulation: {alpha_true}')
print()
print('Note: NB standard errors are larger (Poisson SEs were underestimated).')

In [ ]:
# AIC comparison
print('Model comparison via AIC (lower is better):')
print(f'  Poisson AIC:           {poisson_od.aic:.2f}')
print(f'  Negative Binomial AIC: {nb_model.aic:.2f}')
delta_aic = poisson_od.aic - nb_model.aic
print(f'  ΔAIC (Poisson - NB):   {delta_aic:.2f}')
if delta_aic > 10:
    print('\n  Strong preference for Negative Binomial model.')

In [ ]:
# Visual: compare predicted distributions from both models
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, model, title in zip(
    axes,
    [poisson_od, nb_model],
    ['Poisson Model', 'Negative Binomial Model']
):
    mu_pred = model.predict()
    observed_counts = df_od['y'].values
    ax.scatter(mu_pred, observed_counts, alpha=0.3, s=15, color='steelblue')
    max_val = max(mu_pred.max(), observed_counts.max())
    ax.plot([0, max_val], [0, max_val], 'r--', label='Perfect fit')
    ax.set(xlabel='Predicted μ', ylabel='Observed y',
           title=f'Predicted vs Observed — {title}')
    ax.legend()

plt.tight_layout()
plt.show()

---

## 9. Summary

| Concept | Key takeaway |
|---|---|
| **When to use Poisson regression** | Non-negative integer counts; events per unit of exposure |
| **Link function** | Log link ensures predicted counts are always positive |
| **Coefficient interpretation** | Exponentiate to get Incidence Rate Ratios (IRR) |
| **Offset** | Use `exposure=` when counts come from different exposure windows |
| **Overdispersion** | Check Pearson $\phi = \chi^2_P / df$; $\phi \gg 1$ signals a problem |
| **Negative Binomial** | Adds dispersion parameter $\alpha$; reduces to Poisson when $\alpha=0$ |
| **Model selection** | Compare AIC; NB almost always wins when data are overdispersed |

---

*Next: `13-2-Ejemplo_Poisson.ipynb` — full applied example on a realistic count dataset.*